In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HIQ_H.xpt"
csv_file = DATA_DIR / "HIQ_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Keep completely untouched source for
# exact fidelity comparison.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 HIQ_H XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

#
# NOTE:
#
# HIQ065, HIQ259 and HIQ269 shown in the CDC codebook
# are CHECK ITEMS / routing instructions.
#
# They are NOT released columns in the XPT file.
#

EXPECTED_COLUMNS = [
    "SEQN",
    "HIQ011",
    "HIQ031A",
    "HIQ031B",
    "HIQ031C",
    "HIQ031D",
    "HIQ031E",
    "HIQ031F",
    "HIQ031G",
    "HIQ031H",
    "HIQ031I",
    "HIQ031J",
    "HIQ031AA",
    "HIQ260",
    "HIQ105",
    "HIQ270",
    "HIQ210"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HIQ_H column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 10175:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 17:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,175 rows x 17 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# HIQ_H contains questionnaire response codes only.
#
# There are NO genuine continuous decimal measurements
# documented in this component.
#
# Nevertheless, we prove this from the XPT in Section 6
# BEFORE converting anything.
#

DECIMAL_COLUMNS = []


INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 17:

    raise ValueError(
        "Expected exactly 17 whole-number/code fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Expected genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Expected whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Unlike files such as HUQ, ALQ or DIQ, the CDC HIQ_H
# codebook does NOT document zero as a valid response
# for any released HIQ_H variable.
#
# Therefore we DO NOT automatically restore tiny values
# to zero in this file.
#
# If any tiny values occur, STOP and inspect them.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 17 HIQ_H variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Tiny XPORT values were found in HIQ_H.\n"
        "CDC does not document zero as a valid released "
        "code in these variables, so they must NOT be "
        "automatically converted to zero."
    )


if true_zero_total > 0:

    print(
        "\nOrdinary zeros by variable:"
    )

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected ordinary zeros found in HIQ_H."
    )


print(
    "PASS: No tiny-zero artifacts "
    "and no ordinary zeros."
)


# ============================================================
# 6. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# MUST run before Int64 conversion.
#
# np.round() is used ONLY for detecting fractional values.
# It does NOT alter the dataframe.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values were detected.\n"
        "They will NOT be rounded or converted to Int64:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 7. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83731:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicates."
    )


print(
    "PASS: SEQN 73557-83731 validated."
)


# ============================================================
# 8. CONVERT ALL PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# We already proved above that every populated value
# is a whole number.
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 17 HIQ_H variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 9. HELPER - EXACT CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 10. CDC CHECK - HIQ011
# ============================================================

#
# Covered by health insurance.
#
# 1 = Yes
# 2 = No
# 7 = Refused
# 9 = Don't know
#

check_frequency(
    "HIQ011",
    {
        1: 8594,
        2: 1570,
        7: 3,
        9: 8
    },
    0
)


# ============================================================
# 11. CDC CHECK - HIQ031A
# ============================================================

#
# Private insurance.
#
# 14 = Covered by private insurance
# 77 = Refused
# 99 = Don't know
#

check_frequency(
    "HIQ031A",
    {
        14: 4797,
        77: 1,
        99: 23
    },
    5354
)


# ============================================================
# 12. CDC CHECK - HIQ031B
# ============================================================

check_frequency(
    "HIQ031B",
    {
        15: 1226
    },
    8949
)


# ============================================================
# 13. CDC CHECK - HIQ031C
# ============================================================

check_frequency(
    "HIQ031C",
    {
        16: 11
    },
    10164
)


# ============================================================
# 14. CDC CHECK - HIQ031D
# ============================================================

check_frequency(
    "HIQ031D",
    {
        17: 2402
    },
    7773
)


# ============================================================
# 15. CDC CHECK - HIQ031E
# ============================================================

check_frequency(
    "HIQ031E",
    {
        18: 53
    },
    10122
)


# ============================================================
# 16. CDC CHECK - HIQ031F
# ============================================================

check_frequency(
    "HIQ031F",
    {
        19: 240
    },
    9935
)


# ============================================================
# 17. CDC CHECK - HIQ031G
# ============================================================

check_frequency(
    "HIQ031G",
    {
        20: 10
    },
    10165
)


# ============================================================
# 18. CDC CHECK - HIQ031H
# ============================================================

check_frequency(
    "HIQ031H",
    {
        21: 563
    },
    9612
)


# ============================================================
# 19. CDC CHECK - HIQ031I
# ============================================================

check_frequency(
    "HIQ031I",
    {
        22: 206
    },
    9969
)


# ============================================================
# 20. CDC CHECK - HIQ031J
# ============================================================

check_frequency(
    "HIQ031J",
    {
        23: 233
    },
    9942
)


# ============================================================
# 21. CDC CHECK - HIQ031AA
# ============================================================

#
# 40 = No coverage of any type
#

check_frequency(
    "HIQ031AA",
    {
        40: 8
    },
    10167
)


# ============================================================
# 22. CDC CHECK - HIQ260
# ============================================================

#
# Have Medicare?
#

check_frequency(
    "HIQ260",
    {
        1: 142,
        2: 111,
        7: 2,
        9: 5
    },
    9915
)


# ============================================================
# 23. CDC CHECK - HIQ105
# ============================================================

#
# Insurance card available?
#

check_frequency(
    "HIQ105",
    {
        1: 848,
        2: 86
    },
    9241
)


# ============================================================
# 24. CDC CHECK - HIQ270
# ============================================================

#
# Plans cover prescriptions?
#

check_frequency(
    "HIQ270",
    {
        1: 8236,
        2: 310,
        7: 4,
        9: 53
    },
    1572
)


# ============================================================
# 25. CDC CHECK - HIQ210
# ============================================================

#
# Any time without insurance during past 12 months?
#

check_frequency(
    "HIQ210",
    {
        1: 479,
        2: 8119,
        9: 5
    },
    1572
)


# ============================================================
# 26. ALLOWED CODE VALIDATION
# ============================================================

#
# Explicitly protect against accidental 0 values or
# undocumented codes.
#

ALLOWED_VALUES = {

    "HIQ011": {
        1, 2, 7, 9
    },

    "HIQ031A": {
        14, 77, 99
    },

    "HIQ031B": {
        15
    },

    "HIQ031C": {
        16
    },

    "HIQ031D": {
        17
    },

    "HIQ031E": {
        18
    },

    "HIQ031F": {
        19
    },

    "HIQ031G": {
        20
    },

    "HIQ031H": {
        21
    },

    "HIQ031I": {
        22
    },

    "HIQ031J": {
        23
    },

    "HIQ031AA": {
        40
    },

    "HIQ260": {
        1, 2, 7, 9
    },

    "HIQ105": {
        1, 2
    },

    "HIQ270": {
        1, 2, 7, 9
    },

    "HIQ210": {
        1, 2, 7, 9
    }
}


print("\n--- ALLOWED CODE CHECK ---")


for col, allowed in ALLOWED_VALUES.items():

    observed = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected = (
        observed
        -
        allowed
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected)}"
        )


    print(
        f"PASS: {col} contains only "
        "CDC-documented codes."
    )


# ============================================================
# 27. CONFIRM NO ZERO VALUES EXIST
# ============================================================

#
# This file has no CDC-documented zero response codes.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO CHECK ---")

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected zero values found:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No zero values exist in cleaned HIQ_H."
)


# ============================================================
# 28. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# HIQ_H should require NO numerical correction.
#
# Converting:
#
#     73557.0 -> 73557
#     1.0     -> 1
#     14.0    -> 14
#
# changes presentation only, not numeric meaning.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 17 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 29. EXPORT CSV
# ============================================================

#
# Standard NHANES fidelity export.
#
# NO:
#
#     .round()
#
# NO:
#
#     float_format=
#
# Even though HIQ_H itself has no genuine decimals,
# maintaining this export rule protects the broader
# Dietary/Lab/Examination workflow.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 30. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 31. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 17 CSV variables round-trip "
    "to exact source XPT numeric values."
)


# ============================================================
# 32. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 33. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 34. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 35. RAW CSV .0 CHECK
# ============================================================

#
# HIQ_H contains no genuine decimal variables,
# so checking all 17 columns for unnecessary ".0"
# is safe for THIS file.
#
# Examples:
#
#     SEQN
#     73557.0 -> 73557
#
#     HIQ011
#     1.0 -> 1
#
#     HIQ031A
#     14.0 -> 14
#
#     HIQ031AA
#     40.0 -> 40
#
# Do NOT use this global approach on mixed files
# containing genuine decimal measurements.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any HIQ_H field."
)


# ============================================================
# 36. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO CHECK ---")

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values found "
        "in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no zero values."
)


# ============================================================
# 37. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 38. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83731:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 39. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)


print(
    "HIQ_H CONVERSION COMPLETE"
)


print(
    "=" * 90
)


print(
    "Cycle: 2013-2014"
)


print(
    "Component: Questionnaire - Health Insurance"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Original ordinary zeros:",
    f"{true_zero_total:,}"
)


print(
    "Validated zero restorations: NONE expected"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)


print(
    "HIQ011 insured:",
    int(
        df["HIQ011"]
        .eq(1)
        .sum()
    )
)


print(
    "HIQ031A private insurance:",
    int(
        df["HIQ031A"]
        .eq(14)
        .sum()
    )
)


print(
    "HIQ031B Medicare:",
    int(
        df["HIQ031B"]
        .eq(15)
        .sum()
    )
)


print(
    "HIQ031D Medicaid:",
    int(
        df["HIQ031D"]
        .eq(17)
        .sum()
    )
)


print(
    "HIQ270 prescription coverage yes:",
    int(
        df["HIQ270"]
        .eq(1)
        .sum()
    )
)


print(
    "HIQ210 uninsured sometime past year:",
    int(
        df["HIQ210"]
        .eq(1)
        .sum()
    )
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary '.0': "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2013-2014 HIQ_H XPT -> CSV
Rows: 10,175
Columns: 17
PASS: Structure = 10,175 rows x 17 columns.

--- FIELD TYPE PLAN ---
Expected genuine decimal fields: []
Expected whole-number/code fields: 17

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Ordinary numeric zeros: 0
PASS: No tiny-zero artifacts and no ordinary zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
HIQ011: 0 fractional observations
HIQ031A: 0 fractional observations
HIQ031B: 0 fractional observations
HIQ031C: 0 fractional observations
HIQ031D: 0 fractional observations
HIQ031E: 0 fractional observations
HIQ031F: 0 fractional observations
HIQ031G: 0 fractional observations
HIQ031H: 0 fractional observations
HIQ031I: 0 fractional observations
HIQ031J: 0 fractional observations
HIQ031AA: 0 fractional observations
HIQ260: 0 fractional observations
HIQ105: 0 fractional observations
HIQ270: 0 fractional observations
HIQ210: 0 fractional observations
PASS: No genuine fractional variables.

---